In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Semillas para reproducibilidad
np.random.seed(1)
tf.random.set_seed(1)

In [ ]:
# Datos sintéticos
X, y = make_moons(n_samples=1000, noise=0.25, random_state=1)
X = StandardScaler().fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
def build_model_l2():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(2,),
                              kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.Dense(32, activation='relu',
                              kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

def build_model_dropout():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(2,)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

def build_model_early_stopping():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(2,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

# ===== Configuración y entrenamiento =====
models = {
    'L2 Regularization': build_model_l2,
    'Dropout': build_model_dropout,
    'Early Stopping': build_model_early_stopping
}

histories = {}
for name, builder in models.items():
    model = builder()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    if name == 'Early Stopping':
        callbacks = [tf.keras.callbacks.EarlyStopping(
            patience=5, restore_best_weights=True)]
    else:
        callbacks = []

    h = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=60,
        batch_size=32,
        verbose=0,
        callbacks=callbacks
    )
    histories[name] = h

# ===== Gráficas =====
plt.figure(figsize=(12, 5))

# Precisión
plt.subplot(1, 2, 1)
for name, h in histories.items():
    plt.plot(h.history['accuracy'], label=f'{name} – train')
    plt.plot(h.history['val_accuracy'], '--', label=f'{name} – val')
plt.title('Precisión: entrenamiento vs validación')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Pérdida
plt.subplot(1, 2, 2)
for name, h in histories.items():
    plt.plot(h.history['loss'], label=f'{name} – train')
    plt.plot(h.history['val_loss'], '--', label=f'{name} – val')
plt.title('Pérdida (binary_crossentropy)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()